# 01 - Silver: cleanse, standardise, type and flag

Reads the `brz_*` tables produced by `00_Bronze_Ingest_Excel.ipynb` and writes the Silver layer:

| Table | Grain |
|---|---|
| `slv_pharma_markets` | one row per market |
| `slv_pharma_fx_rates` | month x currency |
| `slv_pharma_products` | one row per product |
| `slv_pharma_customers` | one row per customer |
| `slv_pharma_sales` | one sales transaction (2025 + 2026 unioned across the schema drift) |
| `slv_pharma_forecast` | month x product x market |
| `slv_pharma_budget` | month x product x market |
| `slv_pharma_inventory` | month-end x product x market x warehouse |
| `slv_pharma_opex` | month x market x expense category x scenario |
| `slv_data_quality_summary` | one row per table x data-quality flag |

## The rule this notebook follows

**Silver flags, it does not delete.** Every suspect row keeps a `DataQualityFlags` array, a
`DataQualityFlagCount` and a `DataQualityStatus` of `Valid`, `Review` or `Quarantine`. Gold
decides what to exclude. Deleting here would destroy two things that are *correct*: the negative
quantities (returns) and the large tender orders (outliers).

Duplicates are the one exception, and they are controlled by the `DEDUPE_MODE` parameter.

## What gets done

Trim strings -> standardise capitalisation -> standardise currencies -> parse dates from multiple
text formats -> cast numerics -> reconcile schema drift -> check mandatory columns -> validate
revenue arithmetic -> detect null and orphan keys -> detect duplicates -> classify returns and
outliers -> convert to EUR -> add calculated columns -> write Delta.

**`GrossProfit` is deliberately not created.** It stays a DAX measure
(`[Net Sales] - [Cost]`), as designed.

## 1. Parameters

`DEDUPE_MODE`:
- `"flag"` (default) keeps every row and marks repeats with `IsDuplicateRow = true`
- `"remove"` keeps only the first occurrence of each business key

`FAIL_ON_MISSING_MANDATORY`:
- `False` logs an error and carries on - better while you are still exploring
- `True` raises, which is what you want once this runs on a schedule

In [ ]:
# PARAMETERS -----------------------------------------------------------------
BRONZE_PREFIX = "brz_"
SILVER_PREFIX = "slv_"

DEDUPE_MODE = "flag"                 # "flag" | "remove"
FAIL_ON_MISSING_MANDATORY = False    # True once this is a scheduled pipeline

VALID_CURRENCIES = ["EUR", "GBP", "USD", "CAD", "AED"]
VALID_SCENARIOS  = ["Actual", "Budget", "Forecast"]

# a legitimate return is negative; anything beyond this multiple of the product's
# median order size is flagged for a human to look at, never dropped
OUTLIER_MULTIPLE = 8
# -----------------------------------------------------------------------------

## 2. Imports and session config

`timeParserPolicy = CORRECTED` matters: it makes `to_date` return `NULL` for a value that does not
match the pattern instead of throwing. The whole multi-format date strategy below depends on that
behaviour.

In [ ]:
import re

from pyspark.sql import functions as F
from pyspark.sql import Window as W
from pyspark.sql.types import StringType

for key, value in {
    "spark.sql.legacy.timeParserPolicy": "CORRECTED",   # unmatched date -> NULL, not an error
    "spark.sql.parquet.vorder.enabled": "true",
    "spark.microsoft.delta.optimizeWrite.enabled": "true",
}.items():
    try:
        spark.conf.set(key, value)
    except Exception as exc:
        print(f"[info] could not set {key}: {exc}")

SILVER_TABLES = []      # populated by write_silver(), used by the DQ summary at the end
print("Session ready.")

## 3. Helpers - strings and capitalisation

`clean_text` does the three things every string column needs: trim the outside, collapse repeated
internal whitespace, and turn an empty string into a real `NULL`. That last part is what stops
`""` and `NULL` counting as two different values in a Power BI slicer.

`title_case` is for human-readable values (`germany` -> `Germany`, `HOSPITAL` -> `Hospital`).
`code_case` is for codes that belong in upper case (`eur` -> `EUR`).

In [ ]:
def clean_text(col):
    """Trim, collapse internal whitespace, empty string -> NULL."""
    trimmed = F.regexp_replace(F.trim(col), r"\s+", " ")
    return F.when(trimmed == "", None).otherwise(trimmed)


def trim_all_strings(df):
    """Apply clean_text to every string column in the DataFrame."""
    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            df = df.withColumn(field.name, clean_text(F.col(field.name)))
    return df


def title_case(col):
    """Human-readable values: 'GERMANY' / 'germany ' -> 'Germany'."""
    return F.initcap(F.lower(clean_text(col)))


def code_case(col):
    """Codes: 'eur' / ' Eur ' -> 'EUR'."""
    return F.upper(clean_text(col))


def standardise_against(col, canonical_values):
    """Map a value to its approved spelling by matching on a normalised form.

    initcap is right for prose ('germany' -> 'Germany') but wrong for acronyms: it
    turns 'R&D' into 'R&d', which then splits one department into two dimension rows
    downstream. Where a column has a known set of valid values, match on a normalised
    key and return the approved spelling instead.
    """
    pairs = []
    for value in canonical_values:
        pairs.append(F.lit(re.sub(r"[^a-z0-9]", "", value.lower())))
        pairs.append(F.lit(value))
    lookup = F.create_map(*pairs)
    normalised = F.regexp_replace(F.lower(clean_text(col)), r"[^a-z0-9]", "")
    return F.coalesce(lookup[normalised], clean_text(col))


def standardise_scenario(col):
    """Map any casing/spacing of the scenario to the canonical value."""
    upper = F.upper(clean_text(col))
    return (F.when(upper == "ACTUAL", "Actual")
             .when(upper == "BUDGET", "Budget")
             .when(upper.isin("FORECAST", "FCST"), "Forecast")
             .otherwise(F.initcap(F.lower(upper))))       # unknown value kept, then flagged

print("String helpers ready.")

## 4. Helpers - dates

The source files store dates in several text formats, and in the same column. `parse_any_date`
tries each pattern and takes the first that succeeds.

**Order matters.** `dd/MM/yyyy` is tried before any `MM/dd/yyyy` variant because the slash format
in this source is European. `03-05-2025` in the dash format is American (`MM-dd-yyyy`). Get this
wrong and you will not see an error - you will see the wrong month, silently. That is the single
most dangerous thing in this notebook, which is why the formats are declared in one place.

In [ ]:
DATE_FORMATS = [
    "yyyy-MM-dd",             # ISO - the majority
    "yyyy-MM-dd HH:mm:ss",    # ISO with a time component
    "dd/MM/yyyy",             # European slash format
    "MM-dd-yyyy",             # US dash format
    "dd-MMM-yyyy",            # 05-Mar-2025
    "yyyy/MM/dd",             # ISO with slashes
    "yyyy.MM.dd",             # dotted
    "dd.MM.yyyy",
    "MMM-yyyy",               # month grain: Mar-2026
    "yyyy-MM",                # month grain: 2026-03
    "MM/yyyy",
]


def parse_any_date(col, formats=None):
    """Return the first pattern that parses, else NULL."""
    formats = formats or DATE_FORMATS
    return F.coalesce(*[F.to_date(clean_text(col), fmt) for fmt in formats])


def month_key(date_col):
    """202503 style integer key for joining month-grain facts to a date dimension."""
    return (F.year(date_col) * 100 + F.month(date_col)).cast("int")

print("Date helpers ready.")

## 5. Helpers - numeric casting

Bronze landed everything as text, so every number needs an explicit cast. The regex guard means a
value that is not a number becomes `NULL` rather than throwing, and `to_integer` casts through
`double` so `"2108.0"` survives.

In [ ]:
NUMERIC_PATTERN = r"^-?[0-9]+(\.[0-9]+)?$"


def to_decimal(col, dtype="decimal(18,2)"):
    """Text -> decimal. Strips thousands separators and stray currency symbols."""
    cleaned = F.regexp_replace(clean_text(col), r"[,\s]", "")
    cleaned = F.regexp_replace(cleaned, r"[^0-9.\-]", "")
    return F.when(cleaned.rlike(NUMERIC_PATTERN), cleaned).cast(dtype)


def to_integer(col):
    """Text -> int, tolerating values Excel wrote as '2108.0'."""
    cleaned = F.regexp_replace(clean_text(col), r"[,\s]", "")
    return F.when(cleaned.rlike(NUMERIC_PATTERN), cleaned).cast("double").cast("int")


def to_boolean_flag(col):
    """'Y'/'Yes'/'TRUE'/'1' -> true."""
    upper = F.upper(clean_text(col))
    return F.when(upper.isin("Y", "YES", "TRUE", "1"), F.lit(True)) \
            .when(upper.isin("N", "NO", "FALSE", "0"), F.lit(False))

print("Numeric helpers ready.")

## 6. Helpers - the data-quality flag framework

Every Silver table gets the same three columns, so one Power BI page can report on all of them:

| Column | Meaning |
|---|---|
| `DataQualityFlags` | `array<string>` of everything wrong or notable about the row |
| `DataQualityFlagCount` | `size()` of that array |
| `DataQualityStatus` | `Valid` (no flags), `Review` (informational flags), `Quarantine` (a critical flag) |

Flags are labels, not verdicts. `NEGATIVE_UNITS_RETURN` is informational - the row is correct
business data. Only flags listed as *critical* push a row to `Quarantine`.

In [ ]:
def add_dq_flags(df, rules, critical=()):
    """rules: list of (LABEL, boolean Column). critical: labels that force Quarantine."""
    if rules:
        flag_array = F.array(*[F.when(condition, F.lit(label)) for label, condition in rules])
        flag_array = F.filter(flag_array, lambda x: x.isNotNull())
    else:
        flag_array = F.array().cast("array<string>")

    df = (df.withColumn("DataQualityFlags", flag_array)
            .withColumn("DataQualityFlagCount", F.size("DataQualityFlags")))

    if critical:
        critical_array = F.array(*[F.lit(c) for c in critical])
        has_critical = F.size(F.array_intersect(F.col("DataQualityFlags"), critical_array)) > 0
    else:
        has_critical = F.lit(False)

    return df.withColumn(
        "DataQualityStatus",
        F.when(has_critical, "Quarantine")
         .when(F.col("DataQualityFlagCount") > 0, "Review")
         .otherwise("Valid"))


def mark_duplicates(df, key_cols, order_cols=None, hash_cols=None):
    """Add IsDuplicateRow / DuplicateType / DuplicateKeyCount without dropping anything."""
    order_cols = order_cols or key_cols
    hash_cols = hash_cols or [c for c in df.columns if not c.startswith("_")]

    row_hash = F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("~"))
                                          for c in hash_cols]), 256)
    key_window = W.partitionBy(*key_cols)
    ordered_window = key_window.orderBy(*[F.col(c).asc_nulls_last() for c in order_cols])

    return (df.withColumn("_RowHash", row_hash)
              .withColumn("_Occurrence", F.row_number().over(ordered_window))
              .withColumn("DuplicateKeyCount", F.count(F.lit(1)).over(key_window))
              .withColumn("_DistinctHashes", F.size(F.collect_set("_RowHash").over(key_window)))
              .withColumn("IsDuplicateRow", F.col("_Occurrence") > 1)
              .withColumn("DuplicateType",
                          F.when(F.col("DuplicateKeyCount") == 1, "None")
                           .when(F.col("_DistinctHashes") == 1, "ExactDuplicate")
                           .otherwise("KeyCollision"))
              .drop("_RowHash", "_Occurrence", "_DistinctHashes"))

print("DQ helpers ready.")

## 7. Helpers - schema drift and mandatory columns

`align_to_contract` is the schema-drift engine. It matches source columns to a declared contract by
a **normalised** name - lower-cased with every non-alphanumeric character removed - so
`Unit_Price_Local`, `unitpricelocal` and `CURRENCY` all resolve on their own.

Anything the normaliser cannot resolve goes in the explicit `aliases` dictionary. Columns present
in the contract but absent from the source are added as typed `NULL`s, which is what lets a 2025
file with no `SalesChannel` union cleanly with a 2026 file that has one.

Column **order** is irrelevant to the result - everything is selected back in contract order, so a
`CustomerID` that moved from position 5 to position 11 needs no special handling at all.

In [ ]:
def normalise_name(name):
    """'Unit_Price_Local' / 'CURRENCY' -> 'unitpricelocal' / 'currency'."""
    return re.sub(r"[^a-z0-9]", "", str(name).lower())


def align_to_contract(df, contract, label, aliases=None, mandatory=()):
    """Rename to the contract, add missing columns as NULL, report every difference."""
    lookup = {normalise_name(c): c for c in contract}
    for alias, target in (aliases or {}).items():
        lookup[normalise_name(alias)] = target

    audit_cols = [c for c in df.columns if c.startswith("_")]
    renamed, unmapped = [], []

    for column in list(df.columns):
        if column in audit_cols:
            continue
        target = lookup.get(normalise_name(column))
        if target is None:
            unmapped.append(column)
        elif target != column:
            df = df.withColumnRenamed(column, target)
            renamed.append(f"{column} -> {target}")

    added = [c for c in contract if c not in df.columns]
    for column in added:
        df = df.withColumn(column, F.lit(None).cast("string"))

    print(f"[{label}] schema alignment")
    print(f"    renamed          : {renamed or 'none'}")
    print(f"    added as NULL    : {added or 'none'}")
    print(f"    dropped (unknown): {unmapped or 'none'}")

    missing_mandatory = [c for c in mandatory if c in added]
    if missing_mandatory:
        message = f"[{label}] MANDATORY columns absent from source: {missing_mandatory}"
        if FAIL_ON_MISSING_MANDATORY:
            raise ValueError(message)
        print(f"    [ERROR] {message}")

    return df.select(*contract, *audit_cols)


def finalise(df):
    """Rename Bronze audit columns to Silver naming and stamp the load time."""
    for old, new in [("_SourceFile", "SourceFile"),
                     ("_SourceRowNumber", "SourceRowNumber"),
                     ("_IngestedAtUTC", "BronzeIngestedAtUTC")]:
        if old in df.columns:
            df = df.withColumnRenamed(old, new)
    return df.withColumn("SilverLoadedAtUTC", F.current_timestamp())


def write_silver(df, table_name):
    """Write a Delta table, register it for the DQ summary, print a one-line result."""
    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(table_name))

    written = spark.table(table_name)
    total = written.count()
    if "DataQualityStatus" in written.columns:
        breakdown = {r["DataQualityStatus"]: r["count"]
                     for r in written.groupBy("DataQualityStatus").count().collect()}
    else:
        breakdown = {}
    print(f"  -> {table_name:28s} {total:>7,} rows   {breakdown}")
    if table_name not in SILVER_TABLES:
        SILVER_TABLES.append(table_name)
    return written

print("Schema and write helpers ready.")

---
## 8. `slv_pharma_markets`

Built first because the sales, forecast, budget, inventory and opex tables all use it to recover a
missing `Currency` from the market.

Fixes here: `germany` / `SPAIN` / `France ` / ` Greece` all become proper title case, and
`eur` / `Eur` become `EUR`.

In [ ]:
markets = trim_all_strings(spark.table(BRONZE_PREFIX + "pharma_markets"))

markets = (markets
    .withColumn("MarketID", code_case(F.col("MarketID")))
    .withColumn("Country", title_case(F.col("Country")))
    .withColumn("Region", title_case(F.col("Region")))
    .withColumn("Currency", code_case(F.col("Currency")))
    .withColumn("MarketType", title_case(F.col("MarketType")))
    .withColumn("MarketManager", clean_text(F.col("MarketManager"))))

markets = mark_duplicates(markets, ["MarketID"], ["_SourceRowNumber"])

market_rules = [
    ("MISSING_MARKET_ID",       F.col("MarketID").isNull()),
    ("DUPLICATE_MARKET_ID",     F.col("IsDuplicateRow")),
    ("INVALID_CURRENCY",        F.col("Currency").isNotNull()
                                & ~F.col("Currency").isin(*VALID_CURRENCIES)),
    ("MISSING_CURRENCY",        F.col("Currency").isNull()),
    ("MISSING_MARKET_MANAGER",  F.col("MarketManager").isNull()),
]
markets = add_dq_flags(markets, market_rules,
                       critical=["MISSING_MARKET_ID", "DUPLICATE_MARKET_ID",
                                 "INVALID_CURRENCY", "MISSING_CURRENCY"])

markets = finalise(markets).select(
    "MarketID", "Country", "Region", "Currency", "MarketManager", "MarketType",
    "IsDuplicateRow", "DuplicateType", "DuplicateKeyCount",
    "DataQualityFlags", "DataQualityFlagCount", "DataQualityStatus",
    "SourceFile", "SourceRowNumber", "SilverLoadedAtUTC")

write_silver(markets, SILVER_PREFIX + "pharma_markets")
display(spark.table(SILVER_PREFIX + "pharma_markets")
             .select("MarketID", "Country", "Region", "Currency", "MarketType", "DataQualityFlags"))

---
## 9. `slv_pharma_fx_rates`

`RateToEUR` is *units of local currency per 1 EUR*, so the conversion is
`AmountEUR = AmountLocal / RateToEUR`. Getting this the wrong way round is the classic
multi-currency reporting bug, so the direction is asserted below rather than assumed.

In [ ]:
fx = trim_all_strings(spark.table(BRONZE_PREFIX + "pharma_fx_rates"))

fx = (fx
    .withColumn("RateMonth", parse_any_date(F.col("RateMonth")))
    .withColumn("CurrencyCode", code_case(F.col("CurrencyCode")))
    .withColumn("RateToEUR", to_decimal(F.col("RateToEUR"), "decimal(18,6)"))
    .withColumn("RateType", title_case(F.col("RateType")))
    .withColumn("MonthKey", month_key(F.col("RateMonth"))))

fx = mark_duplicates(fx, ["RateMonth", "CurrencyCode"], ["_SourceRowNumber"])

fx_rules = [
    ("UNPARSEABLE_DATE",   F.col("RateMonth").isNull()),
    ("MISSING_RATE",       F.col("RateToEUR").isNull()),
    ("NON_POSITIVE_RATE",  F.col("RateToEUR") <= 0),
    ("INVALID_CURRENCY",   ~F.col("CurrencyCode").isin(*VALID_CURRENCIES)),
    ("DUPLICATE_RATE_KEY", F.col("IsDuplicateRow")),
]
fx = add_dq_flags(fx, fx_rules, critical=[label for label, _ in fx_rules])

fx = finalise(fx).select(
    "RateMonth", "MonthKey", "CurrencyCode", "RateToEUR", "RateType",
    "IsDuplicateRow", "DuplicateType", "DuplicateKeyCount",
    "DataQualityFlags", "DataQualityFlagCount", "DataQualityStatus",
    "SourceFile", "SourceRowNumber", "SilverLoadedAtUTC")

write_silver(fx, SILVER_PREFIX + "pharma_fx_rates")

# sanity check on the direction of the rate: 1 GBP is worth more than 1 EUR,
# so GBP per EUR must be below 1 and AED per EUR well above it
check = (spark.table(SILVER_PREFIX + "pharma_fx_rates")
              .groupBy("CurrencyCode").agg(F.round(F.avg("RateToEUR"), 4).alias("AvgRateToEUR")))
check.show()

---
## 10. `slv_pharma_products`

`LaunchDate` arrives in three different text formats here - this is the smallest place to prove the
multi-format parser works before trusting it on 5,000 sales rows.

Blank `ProductCategory` is defaulted to `Unclassified` *and* flagged. A dimension with blank
attribute values produces an empty slicer entry in Power BI, which looks like a bug to users.

In [ ]:
products = trim_all_strings(spark.table(BRONZE_PREFIX + "pharma_products"))

products = (products
    .withColumn("ProductID", code_case(F.col("ProductID")))
    .withColumn("ProductName", clean_text(F.col("ProductName")))
    .withColumn("BrandName", clean_text(F.col("BrandName")))
    .withColumn("TherapeuticArea", title_case(F.col("TherapeuticArea")))
    .withColumn("ProductCategoryRaw", F.col("ProductCategory"))
    .withColumn("ProductCategory", F.coalesce(title_case(F.col("ProductCategory")),
                                              F.lit("Unclassified")))
    .withColumn("Manufacturer", clean_text(F.col("Manufacturer")))
    .withColumn("LaunchDate", parse_any_date(F.col("LaunchDate")))
    .withColumn("StandardUnitPriceEUR", to_decimal(F.col("StandardUnitPriceEUR")))
    .withColumn("StandardUnitCostEUR", to_decimal(F.col("StandardUnitCostEUR")))
    .withColumn("PatentExpiryYear", to_integer(F.col("PatentExpiryYear")))
    .withColumn("IsActive", to_boolean_flag(F.col("ActiveFlag"))))

# calculated columns
products = (products
    .withColumn("LaunchYear", F.year("LaunchDate"))
    .withColumn("YearsSinceLaunch",
                F.round(F.months_between(F.current_date(), F.col("LaunchDate")) / 12, 1))
    .withColumn("IsPatentExpired", F.col("PatentExpiryYear") < F.year(F.current_date()))
    .withColumn("ProductLifecycleStage",
                F.when(~F.col("IsActive"), "Discontinued")
                 .when(F.col("IsPatentExpired"), "Post-Patent")
                 .when(F.col("YearsSinceLaunch") < 3, "Launch")
                 .when(F.col("YearsSinceLaunch") < 8, "Growth")
                 .otherwise("Mature")))

products = mark_duplicates(products, ["ProductID"], ["_SourceRowNumber"])

product_rules = [
    ("MISSING_PRODUCT_ID",       F.col("ProductID").isNull()),
    ("DUPLICATE_PRODUCT_ID",     F.col("IsDuplicateRow")),
    ("UNPARSEABLE_LAUNCH_DATE",  F.col("LaunchDate").isNull()),
    ("MISSING_PRODUCT_CATEGORY", F.col("ProductCategoryRaw").isNull()),
    ("MISSING_MANUFACTURER",     F.col("Manufacturer").isNull()),
    ("MISSING_UNIT_PRICE",       F.col("StandardUnitPriceEUR").isNull()),
    ("PRICE_BELOW_COST",         F.col("StandardUnitPriceEUR") < F.col("StandardUnitCostEUR")),
]
products = add_dq_flags(products, product_rules,
                        critical=["MISSING_PRODUCT_ID", "DUPLICATE_PRODUCT_ID",
                                  "UNPARSEABLE_LAUNCH_DATE", "MISSING_UNIT_PRICE",
                                  "PRICE_BELOW_COST"])

products = finalise(products).select(
    "ProductID", "ProductName", "BrandName", "TherapeuticArea", "ProductCategory",
    "Manufacturer", "LaunchDate", "LaunchYear", "YearsSinceLaunch",
    "StandardUnitPriceEUR", "StandardUnitCostEUR", "PatentExpiryYear",
    "IsActive", "IsPatentExpired", "ProductLifecycleStage",
    "IsDuplicateRow", "DuplicateType", "DuplicateKeyCount",
    "DataQualityFlags", "DataQualityFlagCount", "DataQualityStatus",
    "SourceFile", "SourceRowNumber", "SilverLoadedAtUTC")

write_silver(products, SILVER_PREFIX + "pharma_products")
display(spark.table(SILVER_PREFIX + "pharma_products")
             .select("ProductID", "ProductName", "TherapeuticArea", "LaunchDate",
                     "ProductLifecycleStage", "DataQualityFlags"))

---
## 11. `slv_pharma_customers`

Three things worth noting:

- `CustomerType` casing variants (`HOSPITAL`, `hospital`, `Hospital `) collapse to one value.
- A missing `Country` is recovered from the customer's market rather than left blank.
- `CustomerNameNormalised` strips punctuation and casing so the same customer loaded under two
  different IDs can be detected. It is flagged, not merged - deciding the surviving ID is a
  business call, not a pipeline call.

In [ ]:
customers = trim_all_strings(spark.table(BRONZE_PREFIX + "pharma_customers"))

customers = (customers
    .withColumn("CustomerID", code_case(F.col("CustomerID")))
    .withColumn("CustomerName", clean_text(F.col("CustomerName")))
    .withColumn("CustomerType", title_case(F.col("CustomerType")))
    .withColumn("Country", title_case(F.col("Country")))
    .withColumn("MarketID", code_case(F.col("MarketID")))
    .withColumn("CustomerSegmentRaw", F.col("CustomerSegment"))
    .withColumn("CustomerSegment", F.coalesce(title_case(F.col("CustomerSegment")),
                                              F.lit("Unclassified")))
    .withColumn("IsActive", to_boolean_flag(F.col("ActiveFlag"))))

# recover Country from the market dimension where it is missing
market_lookup = (spark.table(SILVER_PREFIX + "pharma_markets")
                      .select(F.col("MarketID").alias("_mkMarketID"),
                              F.col("Country").alias("_mkCountry")))

customers = customers.join(F.broadcast(market_lookup),
                           customers["MarketID"] == F.col("_mkMarketID"), "left")

customers = (customers
    .withColumn("IsCountryDerived", F.col("Country").isNull() & F.col("_mkCountry").isNotNull())
    .withColumn("Country", F.coalesce(F.col("Country"), F.col("_mkCountry")))
    .withColumn("CustomerNameNormalised",
                F.upper(F.regexp_replace(F.coalesce(F.col("CustomerName"), F.lit("")),
                                         r"[^A-Za-z0-9 ]", "")))
    .withColumn("CustomerTypeGroup",
                F.when(F.col("CustomerType") == "Hospital", "Institutional")
                 .when(F.col("CustomerType").isin("Pharmacy Chain", "Retail Pharmacy"), "Retail")
                 .when(F.col("CustomerType").isin("Wholesaler", "Distributor"), "Trade")
                 .otherwise("Other")))

customers = mark_duplicates(customers, ["CustomerID"], ["_SourceRowNumber"])
customers = customers.withColumn(
    "DuplicateNameCount", F.count(F.lit(1)).over(W.partitionBy("CustomerNameNormalised")))

customer_rules = [
    ("MISSING_CUSTOMER_ID",      F.col("CustomerID").isNull()),
    ("DUPLICATE_CUSTOMER_ID",    F.col("IsDuplicateRow")),
    ("DUPLICATE_CUSTOMER_NAME",  F.col("DuplicateNameCount") > 1),
    ("MISSING_MARKET_ID",        F.col("MarketID").isNull()),
    ("ORPHAN_MARKET_ID",         F.col("MarketID").isNotNull() & F.col("_mkMarketID").isNull()),
    ("MISSING_SEGMENT",          F.col("CustomerSegmentRaw").isNull()),
    ("COUNTRY_DERIVED",          F.col("IsCountryDerived")),
    ("UNKNOWN_CUSTOMER_TYPE",    F.col("CustomerTypeGroup") == "Other"),
]
customers = add_dq_flags(customers, customer_rules,
                         critical=["MISSING_CUSTOMER_ID", "DUPLICATE_CUSTOMER_ID",
                                   "MISSING_MARKET_ID", "ORPHAN_MARKET_ID"])

customers = finalise(customers).select(
    "CustomerID", "CustomerName", "CustomerNameNormalised", "CustomerType", "CustomerTypeGroup",
    "CustomerSegment", "Country", "MarketID", "IsActive", "IsCountryDerived",
    "IsDuplicateRow", "DuplicateType", "DuplicateKeyCount", "DuplicateNameCount",
    "DataQualityFlags", "DataQualityFlagCount", "DataQualityStatus",
    "SourceFile", "SourceRowNumber", "SilverLoadedAtUTC")

write_silver(customers, SILVER_PREFIX + "pharma_customers")

print("\nCustomer types after standardisation:")
spark.table(SILVER_PREFIX + "pharma_customers").groupBy("CustomerType").count().show()
print("Customers flagged as a possible duplicate name:")
(spark.table(SILVER_PREFIX + "pharma_customers")
      .filter(F.array_contains("DataQualityFlags", "DUPLICATE_CUSTOMER_NAME"))
      .select("CustomerID", "CustomerName", "MarketID").show(10, truncate=False))

---
## 12. `slv_pharma_sales` - step 1 of 3: schema drift

The two sales files disagree in four ways:

| | 2025 | 2026 update |
|---|---|---|
| price column | `UnitPriceLocal` | `Unit_Price_Local` |
| `CustomerID` | position 5 | position 11 |
| currency header | `Currency` | `CURRENCY` |
| extra columns | - | `SalesChannel`, `SourceSystem` |

The contract below is the target shape. The normalised-name matcher resolves the rename and the
capitalisation change on its own; the missing columns are added as `NULL`; position never matters
because everything is re-selected in contract order.

Run this cell and read the printed alignment report - that report *is* the drift detection.

In [ ]:
SALES_CONTRACT = [
    "TransactionID", "TransactionDate", "ProductID", "MarketID", "CustomerID",
    "UnitsSold", "UnitPriceLocal", "GrossSalesLocal", "DiscountLocal",
    "NetSalesLocal", "CostLocal", "Currency", "ScenarioVersion",
    "SalesChannel", "SourceSystem",
]

# columns the normaliser cannot resolve on its own go here
SALES_ALIASES = {
    "Qty": "UnitsSold",
    "Quantity": "UnitsSold",
    "NetSales": "NetSalesLocal",
    "TxnDate": "TransactionDate",
}

SALES_MANDATORY = ["TransactionID", "TransactionDate", "ProductID", "MarketID",
                   "UnitsSold", "NetSalesLocal", "Currency"]

sales_sources = [
    (BRONZE_PREFIX + "pharma_sales_2025", "Sales 2025"),
    (BRONZE_PREFIX + "pharma_sales_2026", "Sales 2026 update"),
]

aligned = []
for table_name, label in sales_sources:
    source = spark.table(table_name)
    print(f"\n[{label}] source columns ({len(source.columns)}):")
    print("    " + ", ".join(source.columns))
    aligned.append(align_to_contract(source, SALES_CONTRACT, label,
                                     SALES_ALIASES, SALES_MANDATORY))

sales_raw = aligned[0]
for extra in aligned[1:]:
    sales_raw = sales_raw.unionByName(extra, allowMissingColumns=True)

print(f"\nCombined: {sales_raw.count():,} rows, {len(sales_raw.columns)} columns")

## 13. `slv_pharma_sales` - step 2 of 3: standardise, type and enrich

Order matters: parse and cast first, then join the dimensions, because the currency fallback needs
a clean `MarketID` to join on.

The EUR columns are calculated here rather than in DAX. Currency conversion needs a row-level join
to a monthly rate, which is expensive and fragile in a measure - do it once, store both
`*Local` and `*EUR`, and let the report pick.

In [ ]:
sales = trim_all_strings(sales_raw)

# --- types and standardisation ------------------------------------------------
sales = (sales
    .withColumn("TransactionID", code_case(F.col("TransactionID")))
    .withColumn("TransactionDate", parse_any_date(F.col("TransactionDate")))
    .withColumn("ProductID", code_case(F.col("ProductID")))
    .withColumn("MarketID", code_case(F.col("MarketID")))
    .withColumn("CustomerID", code_case(F.col("CustomerID")))
    .withColumn("Currency", code_case(F.col("Currency")))
    .withColumn("ScenarioVersion", standardise_scenario(F.col("ScenarioVersion")))
    .withColumn("SalesChannel", title_case(F.col("SalesChannel")))
    .withColumn("SourceSystem", code_case(F.col("SourceSystem")))
    .withColumn("UnitsSold", to_integer(F.col("UnitsSold")))
    .withColumn("UnitPriceLocal", to_decimal(F.col("UnitPriceLocal")))
    .withColumn("GrossSalesLocal", to_decimal(F.col("GrossSalesLocal")))
    .withColumn("DiscountLocal", to_decimal(F.col("DiscountLocal")))
    .withColumn("NetSalesLocal", to_decimal(F.col("NetSalesLocal")))
    .withColumn("CostLocal", to_decimal(F.col("CostLocal"))))

# --- date parts ---------------------------------------------------------------
sales = (sales
    .withColumn("TransactionYear", F.year("TransactionDate"))
    .withColumn("TransactionMonth", F.month("TransactionDate"))
    .withColumn("MonthStartDate", F.trunc(F.col("TransactionDate"), "MM"))
    .withColumn("MonthKey", month_key(F.col("TransactionDate"))))

# --- recover a missing currency from the market -------------------------------
market_currency = (spark.table(SILVER_PREFIX + "pharma_markets")
                        .select(F.col("MarketID").alias("_mkMarketID"),
                                F.col("Currency").alias("_mkCurrency")))

sales = sales.join(F.broadcast(market_currency),
                   sales["MarketID"] == F.col("_mkMarketID"), "left")

sales = (sales
    .withColumn("IsCurrencyDerived",
                F.col("Currency").isNull() & F.col("_mkCurrency").isNotNull())
    .withColumn("Currency", F.coalesce(F.col("Currency"), F.col("_mkCurrency"))))

# --- convert to EUR on the transaction month's rate ---------------------------
fx_lookup = (spark.table(SILVER_PREFIX + "pharma_fx_rates")
                  .select(F.col("RateMonth").alias("_fxMonth"),
                          F.col("CurrencyCode").alias("_fxCurrency"),
                          "RateToEUR"))

sales = sales.join(F.broadcast(fx_lookup),
                   (F.col("MonthStartDate") == F.col("_fxMonth"))
                   & (F.col("Currency") == F.col("_fxCurrency")), "left")

for local_col in ["UnitPriceLocal", "GrossSalesLocal", "DiscountLocal",
                  "NetSalesLocal", "CostLocal"]:
    sales = sales.withColumn(local_col.replace("Local", "EUR"),
                             F.round(F.col(local_col) / F.col("RateToEUR"), 2))

# NOTE: GrossProfit is deliberately NOT created here.
#       It stays a DAX measure: [Net Sales] - [Cost].

print(f"Standardised {sales.count():,} rows.")
print("Currencies present:",
      [r[0] for r in sales.select("Currency").distinct().orderBy("Currency").collect()])
print("Scenarios present:",
      [r[0] for r in sales.select("ScenarioVersion").distinct().collect()])

## 14. `slv_pharma_sales` - step 3 of 3: validate, flag and write

The validations, and what each one is protecting you from:

| Flag | Critical | What it catches |
|---|---|---|
| `MISSING_PRODUCT_ID` / `MISSING_MARKET_ID` | yes | rows that cannot join a dimension |
| `ORPHAN_PRODUCT_ID` / `ORPHAN_MARKET_ID` | yes | an ID that exists but matches nothing |
| `UNPARSEABLE_DATE` | yes | a date format the parser does not know |
| `GROSS_SALES_MISMATCH` | yes | `Units x Price <> Gross` |
| `NET_SALES_MISMATCH` | yes | `Gross - Discount <> Net` |
| `DUPLICATE_ROW` | yes | the same `TransactionID` loaded twice |
| `INVALID_CURRENCY` / `FX_RATE_MISSING` | yes | the row cannot be converted to EUR |
| `NEGATIVE_UNITS_RETURN` | **no** | a legitimate product return |
| `VOLUME_OUTLIER` | **no** | a genuine bulk or tender order |
| `ZERO_UNITS` | no | a nil line worth a look |

Returns and outliers are `Review`, never `Quarantine`. Dropping them would understate returns and
cut the largest real orders out of the numbers.

In [ ]:
# --- referential integrity ----------------------------------------------------
product_keys = (spark.table(SILVER_PREFIX + "pharma_products")
                     .select(F.col("ProductID").alias("_dimProductID")).distinct())
customer_keys = (spark.table(SILVER_PREFIX + "pharma_customers")
                      .select(F.col("CustomerID").alias("_dimCustomerID")).distinct())

sales = (sales
    .join(F.broadcast(product_keys), sales["ProductID"] == F.col("_dimProductID"), "left")
    .join(F.broadcast(customer_keys), F.col("CustomerID") == F.col("_dimCustomerID"), "left"))

# --- duplicates ---------------------------------------------------------------
sales = mark_duplicates(
    sales,
    key_cols=["TransactionID"],
    order_cols=["_SourceFile", "_SourceRowNumber"],
    hash_cols=["TransactionID", "TransactionDate", "ProductID", "MarketID",
               "CustomerID", "UnitsSold", "NetSalesLocal"])

# --- volume outliers, measured against each product's own median order size ---
median_units = (sales.filter(F.col("UnitsSold") > 0)
                     .groupBy("ProductID")
                     .agg(F.expr("percentile_approx(UnitsSold, 0.5)").alias("_MedianUnits")))

sales = sales.join(F.broadcast(median_units), "ProductID", "left")

# --- calculated columns -------------------------------------------------------
sales = (sales
    .withColumn("IsReturn", F.col("UnitsSold") < 0)
    .withColumn("IsVolumeOutlier",
                F.col("_MedianUnits").isNotNull()
                & (F.abs(F.col("UnitsSold")) > OUTLIER_MULTIPLE * F.col("_MedianUnits")))
    .withColumn("DiscountRatePct",
                F.when(F.col("GrossSalesLocal") != 0,
                       F.round(F.col("DiscountLocal") * 100 / F.col("GrossSalesLocal"), 2))))

# --- validation rules ---------------------------------------------------------
TOLERANCE = 0.01

sales_rules = [
    ("MISSING_TRANSACTION_ID", F.col("TransactionID").isNull()),
    ("MISSING_PRODUCT_ID",     F.col("ProductID").isNull()),
    ("MISSING_MARKET_ID",      F.col("MarketID").isNull()),
    ("MISSING_CUSTOMER_ID",    F.col("CustomerID").isNull()),
    ("ORPHAN_PRODUCT_ID",      F.col("ProductID").isNotNull() & F.col("_dimProductID").isNull()),
    ("ORPHAN_MARKET_ID",       F.col("MarketID").isNotNull() & F.col("_mkMarketID").isNull()),
    ("ORPHAN_CUSTOMER_ID",     F.col("CustomerID").isNotNull() & F.col("_dimCustomerID").isNull()),
    ("UNPARSEABLE_DATE",       F.col("TransactionDate").isNull()),
    ("DUPLICATE_ROW",          F.col("IsDuplicateRow")),
    ("MISSING_UNITS",          F.col("UnitsSold").isNull()),
    ("ZERO_UNITS",             F.col("UnitsSold") == 0),
    ("NEGATIVE_UNITS_RETURN",  F.col("UnitsSold") < 0),
    ("VOLUME_OUTLIER",         F.col("IsVolumeOutlier")),
    ("GROSS_SALES_MISMATCH",
     F.abs(F.col("GrossSalesLocal") - F.col("UnitsSold") * F.col("UnitPriceLocal")) > TOLERANCE),
    ("NET_SALES_MISMATCH",
     F.abs(F.col("NetSalesLocal") - (F.col("GrossSalesLocal") - F.col("DiscountLocal"))) > TOLERANCE),
    # a return has a negative discount by construction - only flag it on a normal sale
    ("NEGATIVE_DISCOUNT",      (F.col("UnitsSold") > 0) & (F.col("DiscountLocal") < 0)),
    ("MISSING_CURRENCY",       F.col("Currency").isNull()),
    ("INVALID_CURRENCY",       F.col("Currency").isNotNull()
                               & ~F.col("Currency").isin(*VALID_CURRENCIES)),
    ("CURRENCY_DERIVED",       F.col("IsCurrencyDerived")),
    ("FX_RATE_MISSING",        F.col("RateToEUR").isNull()),
    ("UNKNOWN_SCENARIO",       ~F.col("ScenarioVersion").isin(*VALID_SCENARIOS)),
]

SALES_CRITICAL = [
    "MISSING_TRANSACTION_ID", "MISSING_PRODUCT_ID", "MISSING_MARKET_ID",
    "ORPHAN_PRODUCT_ID", "ORPHAN_MARKET_ID", "ORPHAN_CUSTOMER_ID",
    "UNPARSEABLE_DATE", "DUPLICATE_ROW", "MISSING_UNITS",
    "GROSS_SALES_MISMATCH", "NET_SALES_MISMATCH",
    "MISSING_CURRENCY", "INVALID_CURRENCY", "FX_RATE_MISSING",
]

sales = add_dq_flags(sales, sales_rules, critical=SALES_CRITICAL)

# --- optional de-duplication --------------------------------------------------
if DEDUPE_MODE == "remove":
    before = sales.count()
    sales = sales.filter(~F.col("IsDuplicateRow"))
    print(f"DEDUPE_MODE='remove': dropped {before - sales.count():,} duplicate rows")
else:
    print("DEDUPE_MODE='flag': duplicates kept and marked with IsDuplicateRow")

sales = finalise(sales).select(
    "TransactionID", "TransactionDate", "TransactionYear", "TransactionMonth",
    "MonthStartDate", "MonthKey",
    "ProductID", "MarketID", "CustomerID", "SalesChannel", "SourceSystem", "ScenarioVersion",
    "UnitsSold", "UnitPriceLocal", "GrossSalesLocal", "DiscountLocal",
    "NetSalesLocal", "CostLocal", "Currency",
    "RateToEUR", "UnitPriceEUR", "GrossSalesEUR", "DiscountEUR", "NetSalesEUR", "CostEUR",
    "DiscountRatePct", "IsReturn", "IsVolumeOutlier", "IsCurrencyDerived",
    "IsDuplicateRow", "DuplicateType", "DuplicateKeyCount",
    "DataQualityFlags", "DataQualityFlagCount", "DataQualityStatus",
    "SourceFile", "SourceRowNumber", "SilverLoadedAtUTC")

write_silver(sales, SILVER_PREFIX + "pharma_sales")

In [ ]:
# What did the validation actually catch?
(spark.table(SILVER_PREFIX + "pharma_sales")
      .select(F.explode("DataQualityFlags").alias("Flag"))
      .groupBy("Flag").count().orderBy(F.desc("count"))
      .show(30, truncate=False))

print("Rows by status:")
spark.table(SILVER_PREFIX + "pharma_sales").groupBy("DataQualityStatus").count().show()

print("Sample of the returns (negative units are correct business data):")
(spark.table(SILVER_PREFIX + "pharma_sales")
      .filter("IsReturn")
      .select("TransactionID", "TransactionDate", "ProductID", "MarketID",
              "UnitsSold", "NetSalesLocal", "Currency")
      .show(5, truncate=False))

---
## 15. `slv_pharma_forecast`

Month-grain, so `Month` arrives in formats the transaction parser never sees (`Mar-2026`,
`2026-03`). Same parser, extra patterns - which is why `DATE_FORMATS` is one shared list.

Blank `Currency` here is recovered from the market, then flagged as `CURRENCY_DERIVED` so nobody
later mistakes a derived value for a sourced one.

In [ ]:
FORECAST_CONTRACT = ["ForecastID", "Month", "ProductID", "MarketID", "UnitsForecast",
                     "RevenueForecastLocal", "CostForecastLocal", "Currency", "ScenarioVersion"]

forecast = align_to_contract(
    spark.table(BRONZE_PREFIX + "pharma_forecast"), FORECAST_CONTRACT, "Forecast 2026",
    aliases={"Units_Forecast": "UnitsForecast"},
    mandatory=["Month", "ProductID", "MarketID", "UnitsForecast", "RevenueForecastLocal"])

forecast = trim_all_strings(forecast)

forecast = (forecast
    .withColumn("ForecastID", code_case(F.col("ForecastID")))
    .withColumn("Month", parse_any_date(F.col("Month")))
    .withColumn("ProductID", code_case(F.col("ProductID")))
    .withColumn("MarketID", code_case(F.col("MarketID")))
    .withColumn("UnitsForecast", to_integer(F.col("UnitsForecast")))
    .withColumn("RevenueForecastLocal", to_decimal(F.col("RevenueForecastLocal")))
    .withColumn("CostForecastLocal", to_decimal(F.col("CostForecastLocal")))
    .withColumn("Currency", code_case(F.col("Currency")))
    .withColumn("ScenarioVersion", F.coalesce(standardise_scenario(F.col("ScenarioVersion")),
                                              F.lit("Forecast")))
    .withColumn("MonthKey", month_key(F.col("Month"))))

market_currency = (spark.table(SILVER_PREFIX + "pharma_markets")
                        .select(F.col("MarketID").alias("_mkMarketID"),
                                F.col("Currency").alias("_mkCurrency")))
forecast = forecast.join(F.broadcast(market_currency),
                         forecast["MarketID"] == F.col("_mkMarketID"), "left")
forecast = (forecast
    .withColumn("IsCurrencyDerived", F.col("Currency").isNull() & F.col("_mkCurrency").isNotNull())
    .withColumn("Currency", F.coalesce(F.col("Currency"), F.col("_mkCurrency"))))

fx_lookup = (spark.table(SILVER_PREFIX + "pharma_fx_rates")
                  .select(F.col("RateMonth").alias("_fxMonth"),
                          F.col("CurrencyCode").alias("_fxCurrency"), "RateToEUR"))
forecast = forecast.join(F.broadcast(fx_lookup),
                         (F.col("Month") == F.col("_fxMonth"))
                         & (F.col("Currency") == F.col("_fxCurrency")), "left")

forecast = (forecast
    .withColumn("RevenueForecastEUR", F.round(F.col("RevenueForecastLocal") / F.col("RateToEUR"), 2))
    .withColumn("CostForecastEUR", F.round(F.col("CostForecastLocal") / F.col("RateToEUR"), 2))
    .withColumn("ForecastUnitPriceLocal",
                F.when(F.col("UnitsForecast") > 0,
                       F.round(F.col("RevenueForecastLocal") / F.col("UnitsForecast"), 2))))

forecast = mark_duplicates(forecast, ["Month", "ProductID", "MarketID"],
                           ["_SourceRowNumber"])

forecast_rules = [
    ("UNPARSEABLE_DATE",     F.col("Month").isNull()),
    ("MISSING_PRODUCT_ID",   F.col("ProductID").isNull()),
    ("MISSING_MARKET_ID",    F.col("MarketID").isNull()),
    ("DUPLICATE_ROW",        F.col("IsDuplicateRow")),
    ("MISSING_UNITS",        F.col("UnitsForecast").isNull()),
    ("NEGATIVE_FORECAST",    F.col("RevenueForecastLocal") < 0),
    ("MISSING_CURRENCY",     F.col("Currency").isNull()),
    ("CURRENCY_DERIVED",     F.col("IsCurrencyDerived")),
    ("FX_RATE_MISSING",      F.col("RateToEUR").isNull()),
    ("UNKNOWN_SCENARIO",     ~F.col("ScenarioVersion").isin(*VALID_SCENARIOS)),
    ("COST_EXCEEDS_REVENUE", F.col("CostForecastLocal") > F.col("RevenueForecastLocal")),
]
forecast = add_dq_flags(forecast, forecast_rules,
                        critical=["UNPARSEABLE_DATE", "MISSING_PRODUCT_ID", "MISSING_MARKET_ID",
                                  "DUPLICATE_ROW", "MISSING_UNITS", "MISSING_CURRENCY",
                                  "FX_RATE_MISSING"])

forecast = finalise(forecast).select(
    "ForecastID", "Month", "MonthKey", "ProductID", "MarketID", "ScenarioVersion",
    "UnitsForecast", "RevenueForecastLocal", "CostForecastLocal", "Currency",
    "RateToEUR", "RevenueForecastEUR", "CostForecastEUR", "ForecastUnitPriceLocal",
    "IsCurrencyDerived", "IsDuplicateRow", "DuplicateType", "DuplicateKeyCount",
    "DataQualityFlags", "DataQualityFlagCount", "DataQualityStatus",
    "SourceFile", "SourceRowNumber", "SilverLoadedAtUTC")

write_silver(forecast, SILVER_PREFIX + "pharma_forecast")

---
## 16. `slv_pharma_budget`

Same grain as the forecast, so the same treatment. The measure names stay distinct
(`BudgetRevenueLocal` vs `RevenueForecastLocal`) because Gold will stack the two into one
`FactPlan` table keyed by `ScenarioVersion` - easier to do there, with both shapes intact.

In [ ]:
BUDGET_CONTRACT = ["Month", "ProductID", "MarketID", "BudgetRevenueLocal",
                   "BudgetCostLocal", "BudgetUnits", "Currency", "ScenarioVersion"]

budget = align_to_contract(
    spark.table(BRONZE_PREFIX + "pharma_budget"), BUDGET_CONTRACT, "Budget 2026",
    mandatory=["Month", "ProductID", "MarketID", "BudgetRevenueLocal", "BudgetUnits"])

budget = trim_all_strings(budget)

budget = (budget
    .withColumn("Month", parse_any_date(F.col("Month")))
    .withColumn("ProductID", code_case(F.col("ProductID")))
    .withColumn("MarketID", code_case(F.col("MarketID")))
    .withColumn("BudgetUnits", to_integer(F.col("BudgetUnits")))
    .withColumn("BudgetRevenueLocal", to_decimal(F.col("BudgetRevenueLocal")))
    .withColumn("BudgetCostLocal", to_decimal(F.col("BudgetCostLocal")))
    .withColumn("Currency", code_case(F.col("Currency")))
    .withColumn("ScenarioVersion", F.coalesce(standardise_scenario(F.col("ScenarioVersion")),
                                              F.lit("Budget")))
    .withColumn("MonthKey", month_key(F.col("Month"))))

budget = budget.join(F.broadcast(market_currency),
                     budget["MarketID"] == F.col("_mkMarketID"), "left")
budget = (budget
    .withColumn("IsCurrencyDerived", F.col("Currency").isNull() & F.col("_mkCurrency").isNotNull())
    .withColumn("Currency", F.coalesce(F.col("Currency"), F.col("_mkCurrency"))))

budget = budget.join(F.broadcast(fx_lookup),
                     (F.col("Month") == F.col("_fxMonth"))
                     & (F.col("Currency") == F.col("_fxCurrency")), "left")

budget = (budget
    .withColumn("BudgetRevenueEUR", F.round(F.col("BudgetRevenueLocal") / F.col("RateToEUR"), 2))
    .withColumn("BudgetCostEUR", F.round(F.col("BudgetCostLocal") / F.col("RateToEUR"), 2))
    .withColumn("BudgetUnitPriceLocal",
                F.when(F.col("BudgetUnits") > 0,
                       F.round(F.col("BudgetRevenueLocal") / F.col("BudgetUnits"), 2))))

budget = mark_duplicates(budget, ["Month", "ProductID", "MarketID"], ["_SourceRowNumber"])

budget_rules = [
    ("UNPARSEABLE_DATE",     F.col("Month").isNull()),
    ("MISSING_PRODUCT_ID",   F.col("ProductID").isNull()),
    ("MISSING_MARKET_ID",    F.col("MarketID").isNull()),
    ("DUPLICATE_ROW",        F.col("IsDuplicateRow")),
    ("MISSING_UNITS",        F.col("BudgetUnits").isNull()),
    ("MISSING_CURRENCY",     F.col("Currency").isNull()),
    ("CURRENCY_DERIVED",     F.col("IsCurrencyDerived")),
    ("FX_RATE_MISSING",      F.col("RateToEUR").isNull()),
    ("UNKNOWN_SCENARIO",     ~F.col("ScenarioVersion").isin(*VALID_SCENARIOS)),
    ("COST_EXCEEDS_REVENUE", F.col("BudgetCostLocal") > F.col("BudgetRevenueLocal")),
]
budget = add_dq_flags(budget, budget_rules,
                      critical=["UNPARSEABLE_DATE", "MISSING_PRODUCT_ID", "MISSING_MARKET_ID",
                                "DUPLICATE_ROW", "MISSING_UNITS", "MISSING_CURRENCY",
                                "FX_RATE_MISSING"])

budget = finalise(budget).select(
    "Month", "MonthKey", "ProductID", "MarketID", "ScenarioVersion",
    "BudgetUnits", "BudgetRevenueLocal", "BudgetCostLocal", "Currency",
    "RateToEUR", "BudgetRevenueEUR", "BudgetCostEUR", "BudgetUnitPriceLocal",
    "IsCurrencyDerived", "IsDuplicateRow", "DuplicateType", "DuplicateKeyCount",
    "DataQualityFlags", "DataQualityFlagCount", "DataQualityStatus",
    "SourceFile", "SourceRowNumber", "SilverLoadedAtUTC")

write_silver(budget, SILVER_PREFIX + "pharma_budget")

print("Duplicate budget keys found:")
(spark.table(SILVER_PREFIX + "pharma_budget")
      .filter("DuplicateKeyCount > 1")
      .select("Month", "ProductID", "MarketID", "BudgetUnits", "DuplicateType")
      .orderBy("Month", "ProductID").show(10, truncate=False))

---
## 17. `slv_pharma_inventory`

The interesting validation here is arithmetic, not formatting:

```
ClosingStock = OpeningStock + UnitsProduced + UnitsReceived - UnitsSold - UnitsExpired
```

`StockVariance` stores the difference so you can see *how far* out a broken row is, not just that
it is broken. A blank `WarehouseID` is defaulted to `UNKNOWN` and flagged.

In [ ]:
INVENTORY_CONTRACT = ["InventoryDate", "ProductID", "MarketID", "WarehouseID",
                      "OpeningStock", "UnitsProduced", "UnitsReceived", "UnitsSold",
                      "UnitsExpired", "ClosingStock", "InventoryValueLocal", "Currency"]

inventory = align_to_contract(
    spark.table(BRONZE_PREFIX + "pharma_inventory"), INVENTORY_CONTRACT, "Inventory",
    mandatory=["InventoryDate", "ProductID", "MarketID", "ClosingStock"])

inventory = trim_all_strings(inventory)

inventory = (inventory
    .withColumn("InventoryDate", parse_any_date(F.col("InventoryDate")))
    .withColumn("ProductID", code_case(F.col("ProductID")))
    .withColumn("MarketID", code_case(F.col("MarketID")))
    .withColumn("WarehouseIDRaw", F.col("WarehouseID"))
    .withColumn("WarehouseID", F.coalesce(code_case(F.col("WarehouseID")), F.lit("UNKNOWN")))
    .withColumn("Currency", code_case(F.col("Currency")))
    .withColumn("InventoryValueLocal", to_decimal(F.col("InventoryValueLocal"))))

for column in ["OpeningStock", "UnitsProduced", "UnitsReceived",
               "UnitsSold", "UnitsExpired", "ClosingStock"]:
    inventory = inventory.withColumn(column, to_integer(F.col(column)))

inventory = (inventory
    .withColumn("MonthKey", month_key(F.col("InventoryDate")))
    .withColumn("MonthStartDate", F.trunc(F.col("InventoryDate"), "MM"))
    .withColumn("ExpectedClosingStock",
                F.col("OpeningStock") + F.col("UnitsProduced") + F.col("UnitsReceived")
                - F.col("UnitsSold") - F.col("UnitsExpired"))
    .withColumn("StockVariance", F.col("ClosingStock") - F.col("ExpectedClosingStock"))
    .withColumn("StockCoverMonths",
                F.when(F.col("UnitsSold") > 0,
                       F.round(F.col("ClosingStock") / F.col("UnitsSold"), 2))))

inventory = inventory.join(F.broadcast(market_currency),
                           inventory["MarketID"] == F.col("_mkMarketID"), "left")
inventory = (inventory
    .withColumn("IsCurrencyDerived", F.col("Currency").isNull() & F.col("_mkCurrency").isNotNull())
    .withColumn("Currency", F.coalesce(F.col("Currency"), F.col("_mkCurrency"))))

inventory = inventory.join(F.broadcast(fx_lookup),
                           (F.col("MonthStartDate") == F.col("_fxMonth"))
                           & (F.col("Currency") == F.col("_fxCurrency")), "left")
inventory = inventory.withColumn(
    "InventoryValueEUR", F.round(F.col("InventoryValueLocal") / F.col("RateToEUR"), 2))

inventory = mark_duplicates(inventory, ["InventoryDate", "ProductID", "MarketID", "WarehouseID"],
                            ["_SourceRowNumber"])

inventory_rules = [
    ("UNPARSEABLE_DATE",        F.col("InventoryDate").isNull()),
    ("MISSING_PRODUCT_ID",      F.col("ProductID").isNull()),
    ("MISSING_MARKET_ID",       F.col("MarketID").isNull()),
    ("MISSING_WAREHOUSE_ID",    F.col("WarehouseIDRaw").isNull()),
    ("DUPLICATE_ROW",           F.col("IsDuplicateRow")),
    ("STOCK_MOVEMENT_MISMATCH", F.col("StockVariance") != 0),
    ("NEGATIVE_CLOSING_STOCK",  F.col("ClosingStock") < 0),
    ("MISSING_CURRENCY",        F.col("Currency").isNull()),
    ("FX_RATE_MISSING",         F.col("RateToEUR").isNull()),
]
inventory = add_dq_flags(inventory, inventory_rules,
                         critical=["UNPARSEABLE_DATE", "MISSING_PRODUCT_ID", "MISSING_MARKET_ID",
                                   "DUPLICATE_ROW", "STOCK_MOVEMENT_MISMATCH",
                                   "NEGATIVE_CLOSING_STOCK", "FX_RATE_MISSING"])

inventory = finalise(inventory).select(
    "InventoryDate", "MonthKey", "MonthStartDate", "ProductID", "MarketID", "WarehouseID",
    "OpeningStock", "UnitsProduced", "UnitsReceived", "UnitsSold", "UnitsExpired",
    "ClosingStock", "ExpectedClosingStock", "StockVariance", "StockCoverMonths",
    "InventoryValueLocal", "Currency", "RateToEUR", "InventoryValueEUR", "IsCurrencyDerived",
    "IsDuplicateRow", "DuplicateType", "DuplicateKeyCount",
    "DataQualityFlags", "DataQualityFlagCount", "DataQualityStatus",
    "SourceFile", "SourceRowNumber", "SilverLoadedAtUTC")

write_silver(inventory, SILVER_PREFIX + "pharma_inventory")

print("Rows where the stock movement does not reconcile:")
(spark.table(SILVER_PREFIX + "pharma_inventory")
      .filter("StockVariance <> 0")
      .select("InventoryDate", "ProductID", "MarketID", "OpeningStock", "UnitsProduced",
              "UnitsReceived", "UnitsSold", "UnitsExpired", "ClosingStock",
              "ExpectedClosingStock", "StockVariance")
      .show(10, truncate=False))

---
## 18. `slv_pharma_opex`

Two judgement calls:

- A blank `Department` is **derived** from the expense category rather than defaulted to
  `Unknown`, because the mapping is one-to-one and known. The row is still flagged as
  `DEPARTMENT_DERIVED` so the fix is auditable.
- Negative amounts are accrual reversals and credit notes. They are marked `IsCreditNote` and left
  alone - filtering them out would overstate costs and silently inflate EBITDA.

In [ ]:
OPEX_CONTRACT = ["Month", "MarketID", "ExpenseCategory", "Department",
                 "AmountLocal", "Currency", "ScenarioVersion"]

opex = align_to_contract(
    spark.table(BRONZE_PREFIX + "pharma_opex"), OPEX_CONTRACT, "Operating expenses",
    aliases={"Amount": "AmountLocal", "Cost_Centre": "Department"},
    mandatory=["Month", "MarketID", "ExpenseCategory", "AmountLocal"])

opex = trim_all_strings(opex)

# Approved spellings. initcap would render 'R&D' as 'R&d' and split the department
# into two rows in the Gold dimension, so these columns are matched against a list.
DEPARTMENT_BY_CATEGORY = {
    "Sales & Marketing": "Commercial",
    "Research & Development": "R&D",
    "General & Administrative": "Corporate Services",
    "Distribution": "Supply Chain",
    "Medical Affairs": "Medical",
    "Regulatory": "Regulatory Affairs",
    "Manufacturing Overhead": "Operations",
}

CANONICAL_CATEGORIES = list(DEPARTMENT_BY_CATEGORY.keys())
CANONICAL_DEPARTMENTS = list(DEPARTMENT_BY_CATEGORY.values())

opex = (opex
    .withColumn("Month", parse_any_date(F.col("Month")))
    .withColumn("MarketID", code_case(F.col("MarketID")))
    .withColumn("ExpenseCategory",
                standardise_against(F.col("ExpenseCategory"), CANONICAL_CATEGORIES))
    .withColumn("DepartmentRaw", F.col("Department"))
    .withColumn("Department",
                standardise_against(F.col("Department"), CANONICAL_DEPARTMENTS))
    .withColumn("AmountLocal", to_decimal(F.col("AmountLocal")))
    .withColumn("Currency", code_case(F.col("Currency")))
    .withColumn("ScenarioVersion", standardise_scenario(F.col("ScenarioVersion")))
    .withColumn("MonthKey", month_key(F.col("Month"))))

category_map = F.create_map(*[F.lit(x) for pair in DEPARTMENT_BY_CATEGORY.items() for x in pair])

opex = (opex
    .withColumn("IsDepartmentDerived", F.col("DepartmentRaw").isNull())
    .withColumn("Department", F.coalesce(F.col("Department"),
                                         category_map[F.col("ExpenseCategory")],
                                         F.lit("Unassigned")))
    .withColumn("IsCreditNote", F.col("AmountLocal") < 0))

opex = opex.join(F.broadcast(market_currency),
                 opex["MarketID"] == F.col("_mkMarketID"), "left")
opex = (opex
    .withColumn("IsCurrencyDerived", F.col("Currency").isNull() & F.col("_mkCurrency").isNotNull())
    .withColumn("Currency", F.coalesce(F.col("Currency"), F.col("_mkCurrency"))))

opex = opex.join(F.broadcast(fx_lookup),
                 (F.col("Month") == F.col("_fxMonth"))
                 & (F.col("Currency") == F.col("_fxCurrency")), "left")
opex = opex.withColumn("AmountEUR", F.round(F.col("AmountLocal") / F.col("RateToEUR"), 2))

opex = mark_duplicates(opex,
                       ["Month", "MarketID", "ExpenseCategory", "Department", "ScenarioVersion"],
                       ["_SourceRowNumber"],
                       hash_cols=["Month", "MarketID", "ExpenseCategory", "Department",
                                  "AmountLocal", "ScenarioVersion"])

opex_rules = [
    ("UNPARSEABLE_DATE",      F.col("Month").isNull()),
    ("MISSING_MARKET_ID",     F.col("MarketID").isNull()),
    ("ORPHAN_MARKET_ID",      F.col("MarketID").isNotNull() & F.col("_mkMarketID").isNull()),
    ("MISSING_CATEGORY",      F.col("ExpenseCategory").isNull()),
    ("DEPARTMENT_DERIVED",    F.col("IsDepartmentDerived")),
    ("DUPLICATE_ROW",         F.col("IsDuplicateRow")),
    ("MISSING_AMOUNT",        F.col("AmountLocal").isNull()),
    ("CREDIT_NOTE",           F.col("IsCreditNote")),
    ("MISSING_CURRENCY",      F.col("Currency").isNull()),
    ("FX_RATE_MISSING",       F.col("RateToEUR").isNull()),
    ("UNKNOWN_SCENARIO",      ~F.col("ScenarioVersion").isin(*VALID_SCENARIOS)),
]
opex = add_dq_flags(opex, opex_rules,
                    critical=["UNPARSEABLE_DATE", "MISSING_MARKET_ID", "ORPHAN_MARKET_ID",
                              "MISSING_CATEGORY", "DUPLICATE_ROW", "MISSING_AMOUNT",
                              "MISSING_CURRENCY", "FX_RATE_MISSING"])

opex = finalise(opex).select(
    "Month", "MonthKey", "MarketID", "ExpenseCategory", "Department", "ScenarioVersion",
    "AmountLocal", "Currency", "RateToEUR", "AmountEUR",
    "IsCreditNote", "IsDepartmentDerived", "IsCurrencyDerived",
    "IsDuplicateRow", "DuplicateType", "DuplicateKeyCount",
    "DataQualityFlags", "DataQualityFlagCount", "DataQualityStatus",
    "SourceFile", "SourceRowNumber", "SilverLoadedAtUTC")

write_silver(opex, SILVER_PREFIX + "pharma_opex")

print("Operating expenses in EUR by scenario:")
(spark.table(SILVER_PREFIX + "pharma_opex")
      .groupBy("ScenarioVersion")
      .agg(F.round(F.sum("AmountEUR") / 1e6, 2).alias("AmountEURm"))
      .orderBy("ScenarioVersion").show())

---
## 19. `slv_data_quality_summary`

One row per table per flag. Point a Power BI page at this table and you have a data-quality
dashboard that covers the whole Silver layer without writing a single extra query.

In [ ]:
summaries = []
for table_name in SILVER_TABLES:
    table = spark.table(table_name)
    if "DataQualityFlags" not in table.columns:
        continue
    summaries.append(
        table.select(F.lit(table_name).alias("SilverTable"),
                     F.explode_outer("DataQualityFlags").alias("DataQualityFlag")))

dq = summaries[0]
for extra in summaries[1:]:
    dq = dq.unionByName(extra)

dq = (dq
    .withColumn("DataQualityFlag", F.coalesce(F.col("DataQualityFlag"), F.lit("(clean row)")))
    .groupBy("SilverTable", "DataQualityFlag")
    .agg(F.count(F.lit(1)).alias("RowCount"))
    .withColumn("ProfiledAtUTC", F.current_timestamp())
    .orderBy("SilverTable", F.desc("RowCount")))

write_silver(dq, SILVER_PREFIX + "data_quality_summary")

dq.filter("DataQualityFlag <> '(clean row)'").show(60, truncate=False)

## 20. Final check

The last number is the one that matters: the share of rows that came through with no flag at all.
On this dataset expect roughly 97-98% for the sales table, which is what the source was built to
produce. A much lower number means a cleaning rule is too aggressive; a much higher one means a
validation is not firing.

In [ ]:
print("Silver layer:")
for table_name in SILVER_TABLES:
    table = spark.table(table_name)
    total = table.count()
    if "DataQualityStatus" in table.columns:
        valid = table.filter("DataQualityStatus = 'Valid'").count()
        quarantine = table.filter("DataQualityStatus = 'Quarantine'").count()
        print(f"  {table_name:30s} {total:>7,} rows | valid {valid:>7,} "
              f"({valid / total:6.1%}) | quarantine {quarantine:>5,}")
    else:
        print(f"  {table_name:30s} {total:>7,} rows")

print("\nSales reconciliation (EUR, excluding quarantined rows):")
(spark.table(SILVER_PREFIX + "pharma_sales")
      .filter("DataQualityStatus <> 'Quarantine'")
      .groupBy("TransactionYear")
      .agg(F.format_number(F.sum("NetSalesEUR") / 1e6, 1).alias("NetSalesEURm"),
           F.format_number(F.sum("CostEUR") / 1e6, 1).alias("CostEURm"),
           F.sum("UnitsSold").alias("Units"))
      .orderBy("TransactionYear").show())

---
## Next

`02_Silver_Validation_Report.ipynb` runs independent checks against these tables - it recalculates
the arithmetic, cross-foots inventory against sales, and proves the schema drift was reconciled
rather than papered over.

Then build Gold: conform the dimensions, union `slv_pharma_forecast` and `slv_pharma_budget` into
one `FactPlan` keyed by `ScenarioVersion`, and filter out `DataQualityStatus = 'Quarantine'` at
that boundary - not before.